In [1]:
from pathlib import Path
import sys
import csv
from collections import defaultdict

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
NOTEBOOKS_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOKS_DIR.parent

DATA_RAW_DIR = PROJECT_ROOT / "DATA" / "RAW"
DATA_PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
SRC_DIR = PROJECT_ROOT / "SRC"

# Support an optional MATCHES subfolder.
if (DATA_RAW_DIR / "MATCHES").is_dir():
    DATA_RAW_DIR = DATA_RAW_DIR / "MATCHES"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_RAW_DIR)
print("Processed data directory:", DATA_PROCESSED_DIR)

assert DATA_RAW_DIR.exists(), "Raw data folder not found."

Project root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE
Raw data directory: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\RAW
Processed data directory: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED


In [3]:
all_csv_files = sorted(DATA_RAW_DIR.glob("*.csv"))

delivery_files = [
    path for path in all_csv_files
    if not path.stem.endswith("_info")
]

info_files = [
    path for path in all_csv_files
    if path.stem.endswith("_info")
]

delivery_ids = {path.stem for path in delivery_files}
info_ids = {path.stem.removesuffix("_info") for path in info_files}

missing_info_ids = sorted(delivery_ids - info_ids)
missing_delivery_ids = sorted(info_ids - delivery_ids)

print("Delivery files:", len(delivery_files))
print("Info files:", len(info_files))
print("Missing info files:", len(missing_info_ids))
print("Missing delivery files:", len(missing_delivery_ids))

if missing_info_ids:
    print("Examples without info files:", missing_info_ids[:10])

if missing_delivery_ids:
    print("Examples without delivery files:", missing_delivery_ids[:10])


Delivery files: 1243
Info files: 1243
Missing info files: 0
Missing delivery files: 0


In [4]:
def parse_match_info(info_path):
    metadata = defaultdict(list)

    with open(info_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)

        for row in reader:
            if not row:
                continue

            if row[0] == "version":
                metadata["version"].append(row[1:])
            elif row[0] == "info" and len(row) >= 3:
                metadata[row[1]].append(row[2:])

    def first_value(key, default=None):
        values = metadata.get(key, [])
        if not values or not values[0]:
            return default
        return values[0][0]

    def integer_value(key):
        value = first_value(key)
        try:
            return int(value) if value is not None else None
        except (TypeError, ValueError):
            return None

    teams = [
        values[0]
        for values in metadata.get("team", [])
        if values
    ]

    players_by_team = defaultdict(list)

    for values in metadata.get("player", []):
        if len(values) >= 2:
            players_by_team[values[0]].append(values[1])

    def metadata_pair_integer(key):
        values = metadata.get(key, [])
        if not values or not values[0]:
            return None

        try:
            return int(values[0][-1])
        except (TypeError, ValueError):
            return None

    record = {
        "match_id": first_value("match_id"),
        "season": first_value("season"),
        "date": first_value("date"),
        "event": first_value("event"),
        "match_type": first_value("match_type"),
        "match_number": first_value("match_number"),
        "gender": first_value("gender"),
        "team_1": teams[0] if len(teams) > 0 else None,
        "team_2": teams[1] if len(teams) > 1 else None,
        "venue": first_value("venue"),
        "city": first_value("city"),
        "toss_winner": first_value("toss_winner"),
        "toss_decision": first_value("toss_decision"),
        "player_of_match": first_value("player_of_match"),
        "winner": first_value("winner"),
        "winner_runs": integer_value("winner_runs"),
        "winner_wickets": integer_value("winner_wickets"),
        "overs": integer_value("overs"),
        "balls_per_over": integer_value("balls_per_over"),
        "target_runs": metadata_pair_integer("target_runs"),
        "target_overs": metadata_pair_integer("target_overs"),
        "team_1_players": (
            " | ".join(players_by_team.get(teams[0], []))
            if len(teams) > 0 else None
        ),
        "team_2_players": (
            " | ".join(players_by_team.get(teams[1], []))
            if len(teams) > 1 else None
        ),
        "umpires": " | ".join(
            values[0]
            for values in metadata.get("umpire", [])
            if values
        ),
        "match_referee": first_value("match_referee"),
        "info_version": first_value("version"),
    }

    return record


In [5]:
delivery_frames = []
match_records = []
failed_delivery_files = []
failed_info_files = []

for delivery_path in delivery_files:
    match_id = delivery_path.stem
    info_path = delivery_path.with_name(f"{match_id}_info.csv")

    if not info_path.exists():
        continue

    try:
        delivery_df = pd.read_csv(delivery_path)

        if delivery_df.empty:
            print("Warning: empty delivery file:", delivery_path.name)

        if "match_id" not in delivery_df.columns:
            raise ValueError("Missing match_id column")

        delivery_frames.append(delivery_df)

    except Exception as exc:
        failed_delivery_files.append({
            "file": delivery_path.name,
            "error": str(exc)
        })
        continue

    try:
        match_records.append(parse_match_info(info_path))

    except Exception as exc:
        failed_info_files.append({
            "file": info_path.name,
            "error": str(exc)
        })

if not delivery_frames:
    raise ValueError("No delivery files loaded successfully.")

deliveries_df = pd.concat(delivery_frames, ignore_index=True)
matches_df = pd.DataFrame(match_records)

print("Delivery rows:", len(deliveries_df))
print("Delivery columns:", len(deliveries_df.columns))
print("Match metadata rows:", len(matches_df))
print("Failed delivery files:", len(failed_delivery_files))
print("Failed info files:", len(failed_info_files))


Delivery rows: 295732
Delivery columns: 27
Match metadata rows: 1243
Failed delivery files: 0
Failed info files: 0


In [6]:

# Delivery data
numeric_delivery_columns = [
    "match_id",
    "season",
    "innings",
    "ball",
    "actual_delivery",
    "runs_off_bat",
    "extras",
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty",
    "non_boundary",
]

for column in numeric_delivery_columns:
    if column in deliveries_df.columns:
        deliveries_df[column] = pd.to_numeric(
            deliveries_df[column], errors="coerce"
        )

if "start_date" in deliveries_df.columns:
    deliveries_df["start_date"] = pd.to_datetime(
        deliveries_df["start_date"], errors="coerce"
    )

# Match metadata
if "match_id" in matches_df.columns:
    matches_df["match_id"] = pd.to_numeric(
        matches_df["match_id"], errors="coerce"
    ).astype("Int64")

if "date" in matches_df.columns:
    matches_df["date"] = pd.to_datetime(
        matches_df["date"], errors="coerce"
    )

for column in [
    "season", "match_number", "winner_runs", "winner_wickets",
    "overs", "balls_per_over", "target_runs", "target_overs"
]:
    if column in matches_df.columns:
        matches_df[column] = pd.to_numeric(
            matches_df[column], errors="coerce"
        ).astype("Int64")

print("Data type standardization complete.")
print("\nDelivery data types:")
display(deliveries_df.dtypes.to_frame("dtype"))

print("\nMatch metadata data types:")
display(matches_df.dtypes.to_frame("dtype"))


Data type standardization complete.

Delivery data types:


,dtype
match_id,int64
season,float64
start_date,datetime64[ns]
venue,object
innings,int64
ball,float64
actual_delivery,float64
batting_team,object
bowling_team,object
striker,object



Match metadata data types:


,dtype
match_id,Int64
season,Int64
date,datetime64[ns]
event,object
match_type,object
match_number,Int64
gender,object
team_1,object
team_2,object
venue,object


In [7]:
delivery_duplicates = deliveries_df.duplicated().sum()
match_duplicates = matches_df.duplicated(subset=["match_id"]).sum()

print("Potential duplicate delivery rows:", delivery_duplicates)
print("Duplicate match metadata IDs:", match_duplicates)

print("\nDuplicate match IDs:")
display(
    matches_df.loc[
        matches_df.duplicated(subset=["match_id"], keep=False)
    ].sort_values("match_id").head(20)
)


Potential duplicate delivery rows: 0
Duplicate match metadata IDs: 0

Duplicate match IDs:


,match_id,season,date,event,match_type,match_number,gender,team_1,team_2,venue,city,toss_winner,toss_decision,player_of_match,winner,winner_runs,winner_wickets,overs,balls_per_over,target_runs,target_overs,team_1_players,team_2_players,umpires,match_referee,info_version


In [8]:
def missing_value_report(df):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2),
    })

    return report.sort_values(
        "missing_percentage", ascending=False
    )

print("DELIVERY MISSING VALUES")
display(missing_value_report(deliveries_df))

print("\nMATCH METADATA MISSING VALUES")
display(missing_value_report(matches_df))


DELIVERY MISSING VALUES


,missing_count,missing_percentage
other_player_dismissed,295732,100.00
fielder_3,295719,100.00
penalty,295730,100.00
other_wicket_type,295732,100.00
non_boundary,295703,99.99
fielder_2,295200,99.82
byes,294987,99.75
noballs,294506,99.59
legbyes,291326,98.51
wides,285856,96.66



MATCH METADATA MISSING VALUES


,missing_count,missing_percentage
winner_runs,685,55.11
winner_wickets,583,46.90
season,178,14.32
match_number,74,5.95
winner,25,2.01
player_of_match,9,0.72
target_overs,7,0.56
target_runs,6,0.48
match_id,0,0.00
date,0,0.00


In [9]:
required_delivery_columns = [
    "match_id",
    "innings",
    "batting_team",
    "bowling_team",
    "striker",
    "bowler",
    "runs_off_bat",
    "extras",
]

missing_columns = [
    column for column in required_delivery_columns
    if column not in deliveries_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required delivery columns are missing: {missing_columns}"
    )

print("Missing required columns:", missing_columns)
print("Invalid innings values:",
      (~deliveries_df["innings"].isin([1, 2, 3, 4])).sum())
print("Negative batter-run values:",
      (deliveries_df["runs_off_bat"] < 0).sum())
print("Negative extras values:",
      (deliveries_df["extras"] < 0).sum())

print("\nDelivery rows by innings:")
display(deliveries_df.groupby("innings").size().rename("delivery_rows"))


Missing required columns: []
Invalid innings values: 12
Negative batter-run values: 0
Negative extras values: 0

Delivery rows by innings:


innings
1    153254
2    142303
3        86
4        77
5         8
6         4
Name: delivery_rows, dtype: int64

In [10]:
delivery_match_ids = set(
    deliveries_df["match_id"].dropna().astype("int64").unique()
)

metadata_match_ids = set(
    matches_df["match_id"].dropna().astype("int64").unique()
)

print("Unique delivery match IDs:", len(delivery_match_ids))
print("Unique metadata match IDs:", len(metadata_match_ids))
print("Delivery matches without metadata:",
      len(delivery_match_ids - metadata_match_ids))
print("Metadata matches without deliveries:",
      len(metadata_match_ids - delivery_match_ids))


Unique delivery match IDs: 1243
Unique metadata match IDs: 1243
Delivery matches without metadata: 0
Metadata matches without deliveries: 0


In [11]:
deliveries_output = DATA_PROCESSED_DIR / "deliveries.csv"
matches_output = DATA_PROCESSED_DIR / "matches.csv"

deliveries_df.to_csv(deliveries_output, index=False)
matches_df.to_csv(matches_output, index=False)

print("Saved files:")
print(deliveries_output)
print(matches_output)

print("\nFinal dataset shapes:")
print("Deliveries:", deliveries_df.shape)
print("Matches:", matches_df.shape)


Saved files:
C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED\deliveries.csv
C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED\matches.csv

Final dataset shapes:
Deliveries: (295732, 27)
Matches: (1243, 26)


In [12]:
cleaning_report = pd.DataFrame([
    {"metric": "delivery_files_found", "value": len(delivery_files)},
    {"metric": "delivery_files_loaded", "value": len(delivery_frames)},
    {"metric": "info_files_found", "value": len(info_files)},
    {"metric": "delivery_rows", "value": len(deliveries_df)},
    {"metric": "delivery_columns", "value": len(deliveries_df.columns)},
    {"metric": "match_rows", "value": len(matches_df)},
    {"metric": "unique_delivery_match_ids", "value": deliveries_df["match_id"].nunique()},
    {"metric": "unique_metadata_match_ids", "value": matches_df["match_id"].nunique()},
    {"metric": "potential_duplicate_delivery_rows", "value": int(delivery_duplicates)},
    {"metric": "duplicate_metadata_ids", "value": int(match_duplicates)},
    {"metric": "failed_delivery_files", "value": len(failed_delivery_files)},
    {"metric": "failed_info_files", "value": len(failed_info_files)},
])

report_path = DATA_PROCESSED_DIR / "cleaning_report.csv"
cleaning_report.to_csv(report_path, index=False)

display(cleaning_report)
print("Cleaning report saved to:", report_path)


,metric,value
0,delivery_files_found,1243
1,delivery_files_loaded,1243
2,info_files_found,1243
3,delivery_rows,295732
4,delivery_columns,27
5,match_rows,1243
6,unique_delivery_match_ids,1243
7,unique_metadata_match_ids,1243
8,potential_duplicate_delivery_rows,0
9,duplicate_metadata_ids,0


Cleaning report saved to: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED\cleaning_report.csv
